<a href="https://colab.research.google.com/github/satani99/triton_kernels/blob/main/fused_softmax.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [19]:
import torch

def naive_softmax(x):
  x_max = x.max(dim=1)[0]

  # print(x.max(dim=1).shape)
  print(x_max.shape)

  safe_x = x - x_max[:, None]
  print(safe_x.shape)

  numerator = torch.exp(safe_x)
  print(numerator.shape)

  denominator = torch.sum(numerator, dim=1)
  print(denominator.shape)

  softmax_out = numerator / denominator[:, None]
  print(softmax_out.shape)

  return softmax_out



In [20]:
!uv pip install triton

Using Python 3.12.13 environment at: /usr
Checked 1 package in 214ms


In [22]:
def softmax(x):
  rows, cols = x.shape
  block_size = triton.next_power_of_2(cols)

  output_buffer = torch.empty_like(x)
  grid = (rows, )

  _softmax_kernel[grid](output_buffer, output_buffer.stride(0), x, x.stride(0), cols, block_size=block_size)
  return output_buffer

In [23]:
import triton
import triton.language as tl
import torch
from torch.nn import functional as F

@triton.jit
def _softmax_kernel(out_ptr, stride_output_row, input_ptr, stride_input_row, num_cols, block_size: tl.constexpr):
  row_index = tl.program_id(0)

  row_start_ptr = input_ptr + (row_index * stride_input_row)
  cols_offset = tl.arange(0, block_size)
  input_pointers = row_start_ptr + cols_offset
  mask = cols_offset < num_cols

  row = tl.load(input_pointers, mask=mask)

  stabilized_row = row - tl.max(row, axis=0)
  numerator = tl.exp(stabilized_row)
  denominator = tl.sum(numerator, axis=0)
  softmax = numerator / denominator

  output_row_start_ptr = out_ptr + row_index * stride_output_row
  output_pointers = output_row_start_ptr + cols_offset
  tl.store(output_pointers, softmax, mask=mask)


In [25]:
x = torch.rand((1000, 1000), device='cuda')
y = softmax(x)